# boto3 + EC2 — A Hands-On Tutorial

This notebook walks through managing **Amazon EC2** (virtual machines) with `boto3`.

**Contents**
1. Setup & imports
2. Client vs. Resource for EC2
3. Discover regions & availability zones
4. Find an AMI (machine image)
5. Key pairs (SSH login)
6. Security groups (firewall rules)
7. Launch an instance (`run_instances`)
8. Describe instances (+ filters)
9. Waiters (wait for state changes)
10. Tags
11. Start / Stop / Reboot / Terminate
12. EBS volumes & Elastic IPs (quick taste)
13. Resource-API style
14. Cleanup
15. Testing **offline** with `moto`
16. **Method reference** — every important method, its parameters, and what it does

> **⚠️ Real money:** running instances, volumes, and Elastic IPs **cost money**.
> Sections 5–14 create real AWS resources and need valid credentials + EC2 permissions.
> Always run the **Cleanup** section. If you just want to explore, use the offline
> `moto` section (15).

## 1. Setup & imports

`pip install boto3` (already in this project). We pin a `REGION` explicitly so results
are predictable.

In [1]:
import os
import boto3
from botocore.exceptions import ClientError

REGION = "us-east-1"
print("boto3 version:", boto3.__version__)

boto3 version: 1.43.32


## 2. Client vs. Resource for EC2

- **Client** (`boto3.client("ec2")`): low-level, mirrors the EC2 API exactly, returns dicts.
- **Resource** (`boto3.resource("ec2")`): object-oriented (`Instance`, `Vpc`, `SecurityGroup`),
  nicer for working with individual objects.

We'll use the **client** for most operations and show the resource style in section 13.

In [2]:
ec2 = boto3.client("ec2", region_name=REGION)
ec2_res = boto3.resource("ec2", region_name=REGION)
print("client:", type(ec2).__name__)
print("resource:", type(ec2_res).__name__)

client: EC2
resource: ec2.ServiceResource


## 3. Discover regions & availability zones

`describe_regions` lists regions you can use; `describe_availability_zones` lists the
AZs within the current region (instances live in a specific AZ).

In [3]:
for r in ec2.describe_regions()["Regions"][:8]:
    print(r["RegionName"], "->", r["Endpoint"])

ap-south-1 -> ec2.ap-south-1.amazonaws.com
eu-north-1 -> ec2.eu-north-1.amazonaws.com
eu-west-3 -> ec2.eu-west-3.amazonaws.com
eu-west-2 -> ec2.eu-west-2.amazonaws.com
eu-west-1 -> ec2.eu-west-1.amazonaws.com
ap-northeast-3 -> ec2.ap-northeast-3.amazonaws.com
ap-northeast-2 -> ec2.ap-northeast-2.amazonaws.com
ap-northeast-1 -> ec2.ap-northeast-1.amazonaws.com


In [4]:
for az in ec2.describe_availability_zones()["AvailabilityZones"]:
    print(az["ZoneName"], az["State"])

us-east-1a available
us-east-1b available
us-east-1c available
us-east-1d available
us-east-1e available
us-east-1f available


## 4. Find an AMI (Amazon Machine Image)

An AMI is the OS/template an instance boots from. AMI IDs differ per region, so look one
up dynamically. Here we grab the latest **Amazon Linux 2023** x86_64 image owned by `amazon`.

In [5]:
images = ec2.describe_images(
    Owners=["amazon"],
    Filters=[
        {"Name": "name", "Values": ["al2023-ami-*-x86_64"]},
        {"Name": "state", "Values": ["available"]},
        {"Name": "architecture", "Values": ["x86_64"]},
    ],
)["Images"]

latest = sorted(images, key=lambda i: i["CreationDate"])[-1]
AMI_ID = latest["ImageId"]
print("Using AMI:", AMI_ID, "-", latest["Name"])

Using AMI: ami-03d84abcde942cf8c - al2023-ami-ecs-hvm-2023.0.20260615-kernel-6.1-x86_64


## 5. Key pairs

A key pair lets you SSH into Linux instances. `create_key_pair` returns the **private key**
material **once** — save it immediately (you can't retrieve it later).

In [6]:
KEY_NAME = "boto-ec2-demo-key"
try:
    kp = ec2.create_key_pair(KeyName=KEY_NAME)
    pem_path = f"{KEY_NAME}.pem"
    with open(pem_path, "w") as f:
        f.write(kp["KeyMaterial"])
    os.chmod(pem_path, 0o400)  # SSH requires tight permissions
    print("Created key pair and saved", pem_path)
except ClientError as e:
    if e.response["Error"]["Code"] == "InvalidKeyPair.Duplicate":
        print("Key pair already exists:", KEY_NAME)
    else:
        raise

Created key pair and saved boto-ec2-demo-key.pem


## 6. Security groups

A security group is a virtual firewall. Create one, then add an **ingress** rule. The
example opens SSH (port 22). In real use, restrict `CidrIp` to your own IP, not `0.0.0.0/0`.

In [7]:
SG_NAME = "boto-ec2-demo-sg"
try:
    sg = ec2.create_security_group(
        GroupName=SG_NAME,
        Description="Demo SG created via boto3",
    )
    SG_ID = sg["GroupId"]
    print("Created security group:", SG_ID)
except ClientError as e:
    if e.response["Error"]["Code"] == "InvalidGroup.Duplicate":
        SG_ID = ec2.describe_security_groups(
            GroupNames=[SG_NAME]
        )["SecurityGroups"][0]["GroupId"]
        print("Security group already exists:", SG_ID)
    else:
        raise

Created security group: sg-0a4955b497e4ff47f


In [8]:
try:
    ec2.authorize_security_group_ingress(
        GroupId=SG_ID,
        IpPermissions=[{
            "IpProtocol": "tcp",
            "FromPort": 22,
            "ToPort": 22,
            "IpRanges": [{"CidrIp": "0.0.0.0/0", "Description": "SSH (demo only)"}],
        }],
    )
    print("Added SSH ingress rule")
except ClientError as e:
    if e.response["Error"]["Code"] == "InvalidPermission.Duplicate":
        print("Ingress rule already present")
    else:
        raise

Added SSH ingress rule


## 7. Launch an instance — `run_instances`

`run_instances` is the core call. `t2.micro` is free-tier eligible. `MinCount`/`MaxCount`
control how many to launch. `TagSpecifications` names it at creation time.

In [9]:
resp = ec2.run_instances(
    ImageId=AMI_ID,
    InstanceType="t2.micro",
    MinCount=1,
    MaxCount=1,
    KeyName=KEY_NAME,
    SecurityGroupIds=[SG_ID],
    TagSpecifications=[{
        "ResourceType": "instance",
        "Tags": [{"Key": "Name", "Value": "boto-ec2-demo"}],
    }],
)
INSTANCE_ID = resp["Instances"][0]["InstanceId"]
print("Launched instance:", INSTANCE_ID)

ClientError: An error occurred (InvalidParameterCombination) when calling the RunInstances operation: The specified instance type is not eligible for Free Tier. For a list of Free Tier instance types, run 'describe-instance-types' with the filter 'free-tier-eligible=true'.

## 8. Describe instances (+ filters)

`describe_instances` returns a nested structure: **Reservations -> Instances**. Use
`Filters` to narrow results (here, only running/pending instances tagged Name=boto-ec2-demo).

In [10]:
resp = ec2.describe_instances(
    Filters=[
        {"Name": "tag:Name", "Values": ["boto-ec2-demo"]},
        {"Name": "instance-state-name", "Values": ["pending", "running"]},
    ]
)
for reservation in resp["Reservations"]:
    for inst in reservation["Instances"]:
        print(inst["InstanceId"], inst["InstanceType"], inst["State"]["Name"],
              inst.get("PublicIpAddress"))

## 9. Waiters

State changes are asynchronous. A **waiter** polls until the instance reaches a state, so
you don't write retry loops. Common EC2 waiters: `instance_running`, `instance_stopped`,
`instance_terminated`, `instance_status_ok`.

In [12]:
waiter = ec2.get_waiter("instance_running")
waiter.wait(InstanceIds=[INSTANCE_ID])
print("Instance is now running:", INSTANCE_ID)

NameError: name 'INSTANCE_ID' is not defined

## 10. Tags

Tags are key/value labels for organizing and finding resources. `create_tags` adds/updates
them on existing resources (or set them at creation via `TagSpecifications`, as in §7).

In [11]:
ec2.create_tags(
    Resources=[INSTANCE_ID],
    Tags=[
        {"Key": "Environment", "Value": "demo"},
        {"Key": "Owner", "Value": "adir"},
    ],
)
print("Tags added")

NameError: name 'INSTANCE_ID' is not defined

## 11. Start / Stop / Reboot / Terminate

- **stop**: shut down but keep the instance (EBS persists; can start again).
- **start**: boot a stopped instance.
- **reboot**: restart in place.
- **terminate**: permanently delete the instance.

In [ ]:
# Stop, then wait until fully stopped.
ec2.stop_instances(InstanceIds=[INSTANCE_ID])
ec2.get_waiter("instance_stopped").wait(InstanceIds=[INSTANCE_ID])
print("Stopped")

# Start again.
ec2.start_instances(InstanceIds=[INSTANCE_ID])
ec2.get_waiter("instance_running").wait(InstanceIds=[INSTANCE_ID])
print("Started")

## 12. EBS volumes & Elastic IPs (quick taste)

Illustrative snippets (commented to avoid extra cost). EBS = block storage disks;
Elastic IP = a static public IPv4 address you can attach to an instance.

In [ ]:
# --- EBS volume ---
# vol = ec2.create_volume(AvailabilityZone="us-east-1a", Size=8, VolumeType="gp3")
# ec2.attach_volume(VolumeId=vol["VolumeId"], InstanceId=INSTANCE_ID, Device="/dev/sdf")

# --- Elastic IP ---
# eip = ec2.allocate_address(Domain="vpc")
# ec2.associate_address(AllocationId=eip["AllocationId"], InstanceId=INSTANCE_ID)
# ec2.release_address(AllocationId=eip["AllocationId"])  # release when done!
print("See the method reference for full parameter details.")

## 13. Resource-API style

The resource API can be more ergonomic for single objects and collections.

In [ ]:
# Same operations, object-oriented:
instance = ec2_res.Instance(INSTANCE_ID)
print("State:", instance.state["Name"], "| Type:", instance.instance_type)

# Filter instances as a collection:
running = ec2_res.instances.filter(
    Filters=[{"Name": "instance-state-name", "Values": ["running"]}]
)
print("Running instance ids:", [i.id for i in running])

## 14. Cleanup

Terminate the instance, wait until gone, then delete the security group and key pair.
**Always run this** so you don't keep paying for resources.

In [ ]:
# Terminate instance.
ec2.terminate_instances(InstanceIds=[INSTANCE_ID])
ec2.get_waiter("instance_terminated").wait(InstanceIds=[INSTANCE_ID])
print("Terminated instance")

# Delete the security group (must be unused first).
try:
    ec2.delete_security_group(GroupId=SG_ID)
    print("Deleted security group")
except ClientError as e:
    print("SG delete:", e.response["Error"]["Code"])

# Delete key pair + local .pem.
ec2.delete_key_pair(KeyName=KEY_NAME)
pem_path = f"{KEY_NAME}.pem"
if os.path.exists(pem_path):
    os.remove(pem_path)
print("Deleted key pair and local .pem")

## 15. Testing **offline** with `moto`

[`moto`](https://docs.getmoto.org/) mocks EC2 in memory — no AWS account, credentials, or
cost. Great for unit tests. Install with `pip install "moto[all]"`.

In [ ]:
try:
    from moto import mock_aws

    @mock_aws
    def demo():
        c = boto3.client("ec2", region_name="us-east-1")
        # moto provides fake AMIs; pick any.
        ami = c.describe_images()["Images"][0]["ImageId"]
        r = c.run_instances(ImageId=ami, InstanceType="t2.micro",
                            MinCount=1, MaxCount=1)
        iid = r["Instances"][0]["InstanceId"]
        print("Mock launched:", iid)

        c.get_waiter("instance_running").wait(InstanceIds=[iid])
        state = c.describe_instances(InstanceIds=[iid])[
            "Reservations"][0]["Instances"][0]["State"]["Name"]
        print("Mock state:", state)

        c.terminate_instances(InstanceIds=[iid])
        print("Mock terminated")

    demo()
except ImportError:
    print('moto not installed. Run: pip install "moto[all]"')

## 16. Method reference

The most important EC2 client methods (`boto3.client("ec2")`). For each: the key
**parameters** (what they represent) and **what the method does**. Parameters marked
*(required)* must be supplied; the rest are common optionals. Most "describe"/action calls
also accept `DryRun` (bool — check permissions without doing the action) and most "describe"
calls accept `Filters`, `MaxResults`, and `NextToken` for pagination.

---

### Lifecycle

**`run_instances(...)`** — launches one or more new EC2 instances.
- `ImageId` *(required)* — the AMI to boot from (the OS/template).
- `InstanceType` — hardware size, e.g. `t2.micro`, `m5.large`.
- `MinCount` *(required)* / `MaxCount` *(required)* — min/max number of instances to launch.
- `KeyName` — name of the SSH key pair for login.
- `SecurityGroupIds` — list of security group IDs (firewall rules) to attach.
- `SubnetId` — the VPC subnet (and thus AZ) to launch into.
- `UserData` — a startup script run on first boot (cloud-init).
- `TagSpecifications` — tags to apply at creation (`ResourceType` + `Tags`).
- `IamInstanceProfile` — IAM role to attach to the instance.
- `BlockDeviceMappings` — disk/EBS configuration.

**`start_instances(InstanceIds=[...])`** — boots previously **stopped** instances.
- `InstanceIds` *(required)* — list of instance IDs to start.

**`stop_instances(InstanceIds=[...], Hibernate=False, Force=False)`** — stops running instances (keeps EBS, can restart).
- `InstanceIds` *(required)* — instances to stop.
- `Hibernate` — if True, suspend to disk instead of a normal stop.
- `Force` — force-stop without a clean shutdown.

**`reboot_instances(InstanceIds=[...])`** — restarts instances in place (like an OS reboot).
- `InstanceIds` *(required)* — instances to reboot.

**`terminate_instances(InstanceIds=[...])`** — permanently deletes instances.
- `InstanceIds` *(required)* — instances to terminate.

---

### Inspection

**`describe_instances(...)`** — returns details about instances, grouped under `Reservations -> Instances`.
- `InstanceIds` — specific instances (omit for all).
- `Filters` — narrow results, e.g. `[{"Name": "instance-state-name", "Values": ["running"]}]`.
- `MaxResults` / `NextToken` — pagination controls.

**`describe_instance_status(...)`** — health/status checks and scheduled events for instances.
- `InstanceIds` — instances to check.
- `IncludeAllInstances` — if True, include non-running instances too (default only running).

**`describe_regions()` / `describe_availability_zones()`** — list regions / AZs available to you.
- `Filters`, `RegionNames`/`ZoneNames` — optional narrowing.

**`describe_images(...)`** — finds AMIs.
- `Owners` — e.g. `["amazon"]`, `["self"]`, or an account ID.
- `Filters` — e.g. by `name`, `architecture`, `state`.
- `ImageIds` — specific AMI IDs.

**`describe_vpcs()` / `describe_subnets()` / `describe_security_groups()`** — list networking resources.
- `Filters`, plus `*Ids`/`GroupNames` to target specific ones.

---

### Key pairs & security groups

**`create_key_pair(KeyName=...)`** — creates an SSH key pair; returns the private key **once**.
- `KeyName` *(required)* — name for the key pair.
- `KeyType` — `rsa` (default) or `ed25519`.

**`delete_key_pair(KeyName=...)`** — deletes a key pair.
- `KeyName` *(required)* — the key pair to delete.

**`create_security_group(GroupName=..., Description=...)`** — creates a firewall group; returns `GroupId`.
- `GroupName` *(required)* — name (unique per VPC).
- `Description` *(required)* — human-readable description.
- `VpcId` — the VPC to create it in (defaults to the default VPC).

**`authorize_security_group_ingress(GroupId=..., IpPermissions=[...])`** — adds inbound (ingress) rules.
- `GroupId` *(required)* — the security group to modify.
- `IpPermissions` — list of rules: `IpProtocol`, `FromPort`, `ToPort`, `IpRanges` (`CidrIp`).

**`authorize_security_group_egress(...)`** — adds outbound (egress) rules (same shape as ingress).

**`revoke_security_group_ingress/egress(...)`** — removes rules (same parameters as authorize).

**`delete_security_group(GroupId=...)`** — deletes a security group (must be unused).
- `GroupId` *(required)* — the group to delete.

---

### Tags

**`create_tags(Resources=[...], Tags=[...])`** — adds/overwrites tags on existing resources.
- `Resources` *(required)* — list of resource IDs (instances, volumes, etc.).
- `Tags` *(required)* — list of `{"Key": ..., "Value": ...}`.

**`delete_tags(Resources=[...], Tags=[...])`** — removes tags from resources.
- `Resources` *(required)* — resource IDs.
- `Tags` — tags to remove (omit value to remove by key).

---

### EBS volumes

**`create_volume(AvailabilityZone=..., Size=...)`** — creates an EBS disk.
- `AvailabilityZone` *(required)* — AZ (must match the instance's AZ to attach).
- `Size` — size in GiB.
- `VolumeType` — `gp3`, `gp2`, `io1`, etc.
- `SnapshotId` — create from an existing snapshot.

**`attach_volume(VolumeId=..., InstanceId=..., Device=...)`** — attaches a volume to an instance.
- `VolumeId` *(required)* — the volume.
- `InstanceId` *(required)* — the target instance.
- `Device` *(required)* — device name, e.g. `/dev/sdf`.

**`detach_volume(VolumeId=...)`** — detaches a volume.
- `VolumeId` *(required)* — the volume to detach.

**`delete_volume(VolumeId=...)`** — deletes a volume (must be detached).
- `VolumeId` *(required)* — the volume to delete.

---

### Elastic IPs (static public addresses)

**`allocate_address(Domain="vpc")`** — allocates a new Elastic IP; returns `AllocationId`.
- `Domain` — `vpc` (standard today).

**`associate_address(AllocationId=..., InstanceId=...)`** — attaches an Elastic IP to an instance.
- `AllocationId` *(required)* — the allocated EIP.
- `InstanceId` — the instance to attach to.

**`disassociate_address(AssociationId=...)`** — detaches an Elastic IP.
- `AssociationId` *(required)* — the association to remove.

**`release_address(AllocationId=...)`** — releases (frees) an Elastic IP. Release unused EIPs — they cost money when not associated.
- `AllocationId` *(required)* — the EIP to release.

---

### Waiters & paginators (helpers, not direct API calls)

**`get_waiter(name)`** — returns a waiter that polls until a state is reached.
- EC2 waiters: `instance_running`, `instance_stopped`, `instance_terminated`, `instance_status_ok`,
  `volume_available`, `volume_in_use`, etc.
- Usage: `ec2.get_waiter("instance_running").wait(InstanceIds=[id])`.

**`get_paginator(operation_name)`** — returns a paginator that iterates all pages of a "describe" call.
- Usage: `for page in ec2.get_paginator("describe_instances").paginate(Filters=[...]): ...`.

> Tip: full parameter lists live in the
> [EC2 client docs](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/ec2.html).